# Анализ алгоритмов

In [4]:
import time
import random

import geopandas as gpd
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from genesis.metrics import *
# from fire_units.metrics import Demand
# from genesis.core import BestPointsBase, StateBase
from genesis.states import FirstArrivalUnitState
from genesis.utils import Progressbar, timing
from genesis.swiss_knife import CMAP_GrGldRd
from genesis.best_points import BestNodeHillClimbing, BestNodesFull, BestNodeMonkey, BestNodesHalfDiameter, BestNodeBee
from genesis.mclp import BestNodesSA

In [5]:
# class Demand(MetricBase):
#     '''
#     Класс-функция расчета метрики удовлетворенности спроса на услуги пожарной охраны
#     '''
#     def __init__(self,
#                  buildings,
#                  f:callable = np.mean,
#                  comp_func:callable = max,
#                  zero_val=0,
#                  buildings_node_id_field: str = 'node',
#                  buildings_demand_field: str = 'demand',
#                  ) -> None:
#         '''
#         `buildings`: pd.GeoDataFrame
#             Геодатафрейм со сведениями о зданиях.
#         `f`: callable
#             Функция расчета показателя. По-умолчанию = np.mean,
#             т.е. вычисляется среднее время следования.
#         `comp_func`: callable
#             Функция сравнения значений метрики.
#             По умолчанию - лучшей считается большая.
#         `zero_val`: float = 0
#             Значение которое будет возвращено в случае передачи набора данных `route_times` без элементов.
#         `buildings_node_id_field`: str = 'node'
#             Имя поля в `buildings` содержащего идентификаторы узлов.
#         buildings_demand_field: str = 'demand'
#             Имя поля в `buildings` содержащего значения спроса
#         '''
#         self.buildings = buildings
#         self.f = f
#         self.zero_val = zero_val
#         self.buildings_node_id_field = buildings_node_id_field
#         self.buildings_demand_field = buildings_demand_field
#         super().__init__(comp_func)

#     def __call__(self, state, area=None):
#         '''
#         `state` (`состояние`): pd.Series
#             Серия времен прибытия в разные точки окружения.

#         `area`: pd.Series
#             Серия данных содержащих маску точек которые должны быть учтены при расчете метрики.
#         '''

#         if not isinstance(state, (list, pd.Series)):
#             raise TypeError(
#                 f"Аргумент state может быть только типа list или pd.Series"
#                 f" Имеется {type(state)}"
#                 )

#         # Отбор узлов по area (списку узлов которые следует учесть в расчете)
#         if not area is None:
#             if isinstance(state, pd.Series):
#                 state_c = state.loc[area]
#             else:
#                 state_c = [x for x in state if x in area]
#         else:
#             state_c = state

#         if len(state_c)==0:
#             return self.zero_val
        
#         # Расчет метрики по спросу
#         merged_df = pd.merge(self.buildings,
#                              state_c,
#                              left_on=self.buildings_node_id_field,
#                              right_index=True)

#         # Расчет собственно состояния удовлетворенности спроса
#         return self.f(merged_df[self.buildings_demand_field] / merged_df['times'])

# Загрузка исходных данных и разбиение по районам

In [6]:
# map_path = 'D:/QGIS/_Красноярск ДИССЕРТАЦИЯ'
map_path = 'G:/QGIS/_Красноярск ДИССЕРТАЦИЯ'

# ГДС
G = ox.load_graphml(map_path+"/fire_data/rng.ml")
print(G.number_of_nodes(), G.number_of_edges())

# районы
districts = gpd.read_file(map_path+"/data/districts.gpkg", layer='Районы')
districts = districts.set_index(districts['name'])
# districts.plot(column='name', legend=True)
print('Районы Красноярска:', list(districts.index))

# здания
buildings = gpd.read_file(map_path+"/data/bld.gpkg", layer='здания')
buildings = buildings[['osmid', 'building', 'kfpo_spros', 'geometry']]   # 'name', 'amenity', 
buildings = buildings.set_index('osmid')
print('Всего зданий в Красноярске:', len(buildings))

31293 77344
Районы Красноярска: ['Железнодорожный район', 'Кировский район', 'Ленинский район', 'Октябрьский район', 'Свердловский район', 'Советский район', 'Центральный район']
Всего зданий в Красноярске: 58094


Установка скоростей следования для ГДС

In [7]:
from graphs.speeds import kmh_to_mm, set_graph_travel_times

speeds = [53, 40, 28.1, 17.3, 5.4]
speeds = [kmh_to_mm(s) for s in speeds]

set_graph_travel_times(G, speeds)

g:\GitRepositories\_Dislocation\genesis\work\genesis\graphs\speeds.py:96: UserWarning: Граф был ранее упрощен! Оценка времени следования такого графа может повлечь неточности. Рекомендуется устанавливать время следования для неупрощенного графа и только после этого упрощать
  warnings.warn("Граф был ранее упрощен! Оценка времени следования такого графа может повлечь неточности. " + \


Приведение к единой системе координат

In [8]:
print('Проверка СК:', end=' ')
if not G.graph['crs'] == districts.crs == buildings.crs:
    raise ValueError('CRS G, districts и buildings различны!')
else:
    print('CRS совпадают.')

Проверка СК: CRS совпадают.


In [9]:
G = ox.project_graph(G)
districts = ox.project_gdf(districts)
buildings = ox.project_gdf(buildings)

print('Проверка СК:', end=' ')
if not G.graph['crs'] == districts.crs == buildings.crs:
    raise ValueError('CRS G, districts и buildings различны!')
else:
    print('CRS совпадают.')

Проверка СК: CRS совпадают.


Дополнение зданий сведениями о ближайших узлах ГДС

In [10]:
g_nodes = ox.graph_to_gdfs(G, edges=False)
buildings = buildings.sjoin_nearest(g_nodes[['geometry']], max_distance=100)
buildings.rename(columns={'index_right': 'node'}, inplace=True)

Разбор графа и зданий на наборы для районов

In [11]:
# g_nodes = ox.graph_to_gdfs(G, edges=False)

scopes = {}
for i, district in districts.iterrows():
    poly = district['geometry']
    print(f'{i}: ', end='')

    # Граф
    g_nodes_in = g_nodes.loc[g_nodes.intersects(poly.buffer(1000))]
    g = nx.subgraph(G, g_nodes_in.index)
    # оставляем только больший сильносвязный компонент:
    # g = max(nx.strongly_connected_components(g), key=len)
    g = nx.subgraph(g, max(nx.strongly_connected_components(g), key=len))
    print(g.number_of_nodes(), g.number_of_edges(), end=' ')

    # Здания
    b = buildings.loc[buildings.intersects(poly)]
    print(len(b))

    scopes[i] = [poly, g, b]

Железнодорожный район: 3515 8411 2898
Кировский район: 5100 12939 3382
Ленинский район: 5953 15022 4812
Октябрьский район: 7008 17201 11798
Свердловский район: 5490 13597 15159
Советский район: 9377 22942 8101
Центральный район: 5641 13763 6440


In [ ]:
# ДЛЯ ПЕЧАТИ КАРТЫ РАСКОММЕНТИРОВАТЬ!
# def plot_g(name, g, b, poly, ax):
#     ax.set_title(name)
#     ax.set_axis_off()
#     ax.set_facecolor('white')
#     ox.plot_graph(g, node_size=0, edge_linewidth=0.2, ax=ax, show=False)
#     b.plot(ax=ax)
#     poly.plot(color='none', edgecolor='r', linewidth=1, ax=ax)


# fig, axs = plt.subplots(3,3, tight_layout=True, figsize=(12,12), dpi=300)
# x,y = 0,0
# for district in districts.index:
#     print(f'{district} [{y,x}]: ', end='')
#     g = scopes[district][1]
#     b = scopes[district][2]
#     plot_g(district, g, b, districts.loc[[district]], axs[y,x])
#     x += 1
#     if x >= 3:
#         x = 0
#         y += 1
#     print('ok')

# ax=axs[2,1]
# ax.set_title('г. Красноярск')
# ax.set_axis_off()
# ax.set_facecolor('white')
# districts.plot(column='name', ax=ax, legend=True, legend_kwds={'fontsize': 6, 'markerscale':0.5})

# plt.delaxes(axs[2,2])

# fig.tight_layout()

# Вычислительные эксперименты

## BLP

In [12]:
from genesis.utils import TimeCheckPoint

# class TimeCheckPoint(object):
#     '''
#     Таймер

#     `acc`:int
#         Точность округления
#     '''
#     def __init__(self, acc:int=2):
#         self.acc = acc
#         self.prev = time.time()
    
#     def __call__(self, switch=True):
#         now = time.time()
#         diff = round(now - self.prev, self.acc)
#         if switch:
#             self.prev = now
#         return diff

# t = TimeCheckPoint()

# time.sleep(1)
# print(t(False), end=', ')
# # print(t())
# time.sleep(2)
# print(t(False), end=', ')
# # print(t())
# time.sleep(3)
# print(t(False), end=', ')
# # print(t())
# time.sleep(4)
# print(t(False), end=', ')
# # print(t())


In [13]:
from fire_units.metrics import Demand


main_times_list = {}
main_accuracy_list = {}
main_accuracy2_list = {}

for district, scope in scopes.items():
    print(' ')
    print('='*80)
    print(district)
    g = scopes[district][1]
    b = scopes[district][2]
    nodes = ox.graph_to_gdfs(g, edges=False)
    area_nodes = nodes.within(scopes[district][0])
    nodes_list=set(nodes[area_nodes].index)
    print('Узлов', sum(area_nodes))
    print('='*80)

    # функция метрики спроса
    metric_demand = Demand(b, buildings_demand_field='kfpo_spros')

    # Списки результатов расчета
    times_list = {}
    accuracy_list = {}
    accuracy2_list = {}     # Для метрики по результату

    # Расчет полным перебором
    # ============================================================================
    print('\n', '# Расчет полным перебором')
    print('-'*80)
    t = TimeCheckPoint()
    t2 = TimeCheckPoint()
    pb = Progressbar(sum(area_nodes), bins=40)
    full = BestNodesFull(FirstArrivalUnitState(), metric_demand, node_calc_end_function=pb)
    best_nodes, best_metric = full(env=g, area=area_nodes, nodes_list=nodes_list)
    time_spent = t()

    # Сохраняем результат вычисления
    times_list['Полный перебор'] = time_spent
    accuracy_list['Полный перебор'] = 100.0
    accuracy2_list['Полный перебор'] = 100.0

    # Сохраняем эталонные значения
    etalon_node, etalon_metric, etalon_time = best_nodes[0], best_metric, time_spent

    # Расчет HC
    # ============================================================================
    print('\n', '# Расчет HC')
    print('-'*80)
    bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)
    bn_list = []
    bn_metrics = []
    i=1
    t()
    while t(False) < etalon_time:
        t2()
        start_node = random.choice(list(nodes_list))
        best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
        bn_list.append(best_node)
        bn_metrics.append(best_metric)
        print('\n', i, best_node, best_metric, t2(), end='\r')
        i+=1
    time_spent = t()

    # Сохраняем результат вычисления
    correct_results = bn_list.count(etalon_node)
    times_list['HC'] = round(time_spent/len(bn_list), 1)
    accuracy_list['HC'] = round(100*correct_results/len(bn_list), 1)
    accuracy2_list['HC'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
    print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["HC"]}', '|', accuracy2_list['HC'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["HC"]} сек')

    # Расчет MSA
    # ============================================================================
    print('\n', '# Расчет MSA')
    print('-'*80)
    class EventFunc:
        def __init__(self):
            pass
        def __call__(self, **kwargs):
            global_jump_number = kwargs['global_jump_number']
            local_jump_number = kwargs.get('local_jump_number', 0)
            print("Глобальный прыжок:", global_jump_number, ", Локальный прыжок:", local_jump_number, '  ', end='\r')

    bnmsa = BestNodeMonkey(FirstArrivalUnitState(), metric_demand, 
                                global_jumps_count=2,
                                local_jumps_count=5,
                                all_neighbors=True,
                                after_global_jump_function=EventFunc(),
                                after_local_jump_function=EventFunc(),
                                )
    bn_list = []
    bn_metrics = []
    i=1
    t()
    while t(False) < etalon_time:
        t2()
        best_node, best_metric = bnmsa(env=g, area=area_nodes)
        bn_list.append(best_node)
        bn_metrics.append(best_metric)
        print('\n', i, best_node, best_metric, t2(), end='\r')
        i+=1
    time_spent = t()

    # Сохраняем результат вычисления
    correct_results = bn_list.count(etalon_node)
    times_list['MSA'] = round(time_spent/len(bn_list), 1)
    accuracy_list['MSA'] = round(100*correct_results/len(bn_list),1)
    accuracy2_list['MSA'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
    print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["MSA"]}', '|', accuracy2_list['MSA'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["MSA"]} сек')

    # Расчет HC+BNHD
    # ============================================================================
    print('\n', '# Расчет HC+BNHD')
    print('-'*80)
    bnhd = BestNodesHalfDiameter(FirstArrivalUnitState(), metric_demand)

    t()
    start_node = bnhd(env=g, area=area_nodes)
    best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
    time_spent = t()

    # Сохраняем результат вычисления
    correct_results = best_metric/etalon_metric
    times_list['HC+BNHD'] = time_spent
    accuracy_list['HC+BNHD'] = round(100*correct_results,1)
    accuracy2_list['HC+BNHD'] = round(100*correct_results,1)
    print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["HC+BNHD"]}', '|', accuracy2_list['HC+BNHD'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["HC+BNHD"]} сек')


    # Расчет MSA+BNHD
    # ============================================================================
    print('\n', '# Расчет MSA+BNHD')
    print('-'*80)
    bnmsa = BestNodeMonkey(FirstArrivalUnitState(), metric_demand, 
                               global_jumps_count=2,
                               local_jumps_count=10,
                               all_neighbors=True,
                               local_jump_max_distance=1500,
                               after_global_jump_function=EventFunc(),
                               after_local_jump_function=EventFunc(),
                               )

    bn_list = []
    bn_metrics = []
    i=1
    t()
    while t(False) < etalon_time:
        t2()
        start_node = bnhd(env=g, area=area_nodes)
        best_node, best_metric = bnmsa(env=g, area=area_nodes, start_node=start_node)
        bn_list.append(best_node)
        bn_metrics.append(best_metric)
        print('\n', i, best_node, best_metric, t2(), end='\r')
        i+=1
    time_spent = t()

    # Сохраняем результат вычисления
    correct_results = bn_list.count(etalon_node)
    times_list['MSA+BNHD'] = round(time_spent/len(bn_list), 1)
    accuracy_list['MSA+BNHD'] = round(100*correct_results/len(bn_list),1)
    accuracy2_list['MSA+BNHD'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
    print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["MSA+BNHD"]}', '|', accuracy2_list['MSA+BNHD'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["MSA+BNHD"]} сек')






    # Расчет ABC
    # ============================================================================
    print('\n', '# Расчет ABC')
    print('-'*80)
    bnabc = BestNodeBee(FirstArrivalUnitState(),
                    metric_demand,
                    # scouts_count=150,
                    # best_scouts_count=3,
                    )
    bn_list = []
    bn_metrics = []
    i=1
    t()
    while t(False) < etalon_time:
        t2()
        best_node, best_metric = bnabc(env=g, area=area_nodes, nodes_list=nodes_list)
        bn_list.append(best_node)
        bn_metrics.append(best_metric)
        print('\n', i, best_node, best_metric, t2(), end='\r')
        i+=1
    time_spent = t()

    # Сохраняем результат вычисления
    correct_results = bn_list.count(etalon_node)
    times_list['ABC'] = round(time_spent/len(bn_list), 1)
    accuracy_list['ABC'] = round(100*correct_results/len(bn_list),1)
    accuracy2_list['ABC'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
    print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["ABC"]}', '|', accuracy2_list['ABC'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["ABC"]} сек')


    # Расчет SA
    # ============================================================================
    print('\n', '# Расчет SA')
    print('-'*80)

    def best_state_find_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
        print(f'\nШаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |')

    mclpsa = BestNodesSA(FirstArrivalUnitState(),
                        metric_demand,
                        end_temperature=0.0001,
                        turns= 1000,
                        #  turn_end_function=turn_end_function,
                        best_state_find_function=best_state_find_function,
                        )

    t()
    # best_node, best_metric = mclpsa(env=g, area=area_nodes, start_node=start_node)
    best_node, best_metric = mclpsa(env=g, area=area_nodes, dynamic_nodes={list(g.nodes())[0]:'best'})
    time_spent = t()

    # Сохраняем результат вычисления
    correct_results = best_metric/etalon_metric
    times_list['SA'] = time_spent
    accuracy_list['SA'] = round(100*correct_results,1)
    accuracy2_list['SA'] = round(100*correct_results,1)
    print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["SA"]}', '|', accuracy2_list['SA'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["SA"]} сек')

    # Сохранение итогов в основные словари:
    main_times_list[district+' время'] = times_list
    main_accuracy_list[district+' точность'] = accuracy_list
    main_accuracy2_list[district+' точность'] = accuracy2_list
    # Промежуточное сохранение в DataFrame с объединением main_times_list и main_accuracy_list
    # ------------------------------
    # tdf_times = pd.DataFrame(main_times_list)
    # tdf_times.to_csv('times.csv')
    # tdf_accuracy = pd.DataFrame(main_accuracy_list)
    # tdf_accuracy.to_csv('accuracy.csv')
    tdf = pd.DataFrame({**main_times_list, **main_accuracy_list, **main_accuracy2_list})
    tdf.to_csv('metrics.csv')
    tdf.to_csv(f'{district}.csv')

 
Железнодорожный район
Узлов 1861

 # Расчет полным перебором
--------------------------------------------------------------------------------
|########################################| 100.0%

 # Расчет HC
--------------------------------------------------------------------------------

 1 346120277 0.0010154349277178602 7.87
 2 346119674 0.0010675498552995552 0.73
 3 346030545 0.0010219066494003804 2.05
 4 346020740 0.001077479766364616 2.7
 5 346119674 0.0010675498552995552 9.66
 6 4550836107 0.0008644799707844963 5.88
 7 346119674 0.0010675498552995552 3.13
 8 346020740 0.001077479766364616 2.07
 9 346119674 0.0010675498552995552 4.5
 10 345759063 0.00111746411505084 2.15
 11 346120277 0.0010154349277178602 4.67
 12 4550836107 0.0008644799707844963 11.14
 13 345347119 0.0009867546021765203 1.93
 14 1937597514 0.0009345301046902013 1.13
 15 290662851 0.0010715280868907012 1.02
 16 346020740 0.001077479766364616 3.95
 17 346119674 0.0010675498552995552 11.63
 18 1937597514 0.0009345

In [ ]:
main_times_list

In [ ]:
# Сохранение итогов в основные словари:
main_times_list = {}
main_accuracy_list = {}
main_times_list[district+' время'] = times_list
main_accuracy_list[district+' точность'] = accuracy_list
# Промежуточное сохранение в DataFrame с объединением main_times_list и main_accuracy_list
# ------------------------------
# tdf_times = pd.DataFrame(main_times_list)
# tdf_times
# tdf_times.to_csv('times.csv')
# tdf_accuracy = pd.DataFrame(main_accuracy_list)
# tdf_accuracy.to_csv('accuracy.csv')
pd.DataFrame({**main_times_list, **main_accuracy_list})

### Черновики

In [ ]:
district = 'Железнодорожный район'
poly = scopes[district][0]
g    = scopes[district][1]
b    = scopes[district][2]
nodes = ox.graph_to_gdfs(g, edges=False)
area_nodes = nodes.within(poly)
nodes_list=set(nodes[area_nodes].index)

print('Узлов', sum(area_nodes))

# функция метрики спроса
metric_demand = Demand(b, buildings_demand_field='kfpo_spros')

In [ ]:
# Полный перебор - эталонное значение
pb = Progressbar(sum(area_nodes), bins=40)
full = timing(BestNodesFull(FirstArrivalUnitState(), metric_demand, node_calc_end_function=pb), return_time=True)

best_nodes, best_metric, time_spent = full(env=g, area=area_nodes, nodes_list=nodes_list)
etalon_node, etalon_metric, etalon_time = best_nodes[0], best_metric, time_spent

etalon_node, etalon_metric, etalon_time

In [ ]:
etalon_node, etalon_metric, etalon_time = 345759063, 0.00111746411505084, 344.15

In [ ]:
bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)

bn_list = []
bn_metrics = []
i=1

ts = time.time()
while time.time() - ts < etalon_time:
    tcs = time.time()
    start_node = random.choice(list(nodes_list))
    best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
    bn_list.append(best_node)
    bn_metrics.append(best_metric)
    tcd = time.time() - tcs
    print('\n', i, best_node, best_metric, round(tcd, 2), end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

In [ ]:
class EventFunc:
    def __init__(self):
        pass
    
    def __call__(self, **kwargs):
        global_jump_number = kwargs['global_jump_number']
        local_jump_number = kwargs.get('local_jump_number', 0)
        print("Глобальный прыжок:", global_jump_number, ", Локальный прыжок:", local_jump_number, '  ', end='\r')

bnmsa = BestNodeMonkey(FirstArrivalUnitState(), metric_demand, 
                               global_jumps_count=2,
                               local_jumps_count=5,
                               all_neighbors=True,
                               after_global_jump_function=EventFunc(),
                               after_local_jump_function=EventFunc(),
                               )

bn_list = []
bn_metrics = []
i=1

ts = time.time()
while time.time() - ts < etalon_time:
    tcs = time.time()
    best_node, best_metric = bnmsa(env=g, area=area_nodes)
    bn_list.append(best_node)
    bn_metrics.append(best_metric)
    tcd = time.time() - tcs
    print('\n', i, best_node, best_metric, round(tcd, 2), end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')


best_node, best_metric

In [ ]:
# HC+BNHD
bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)
bnhd = BestNodesHalfDiameter(FirstArrivalUnitState(), metric_demand)

ts = time.time()

start_node = bnhd(env=g, area=area_nodes)
best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)

tf = time.time()
td = round(tf-ts, 2)

print(f'ВРЕМЯ: {td} сек')

correct_results = best_metric/etalon_metric
print(f'КОРРЕКТНОСТЬ: {correct_results}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td} сек')

In [ ]:
# MSA+BNHD
bnhd = BestNodesHalfDiameter(FirstArrivalUnitState(), metric_demand)
bnmsa = BestNodeMonkey(FirstArrivalUnitState(), metric_demand, 
                               global_jumps_count=2,
                               local_jumps_count=10,
                               all_neighbors=True,
                               local_jump_max_distance=1500,
                               after_global_jump_function=EventFunc(),
                               after_local_jump_function=EventFunc(),
                               )

ts = time.time()

start_node = bnhd(env=g, area=area_nodes)
# best_node, best_metric = bnmsa(env=g, area=area_nodes, start_node=start_node)

bn_list = []
bn_metrics = []
i=1

# while time.time() - ts < etalon_time:
while i < 6:
    tcs = time.time()
    best_node, best_metric = bnmsa(env=g, area=area_nodes, start_node=start_node)
    bn_list.append(best_node)
    bn_metrics.append(best_metric)
    tcd = time.time() - tcs
    print('\n', i, best_node, best_metric, round(tcd, 2), end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

best_node, best_metric

In [ ]:
# ABC
from genesis.best_points import BestNodeBee

bnabc = BestNodeBee(FirstArrivalUnitState(),
                    metric_demand,
                    # scouts_count=150,
                    # best_scouts_count=3,
                    )

ts = time.time()
bn_list = []
bn_metrics = []
i=1
while time.time() - ts < etalon_time:
# while i < 5:
    print(i, end=':')
    tcs = time.time()

    best_node, best_metric = bnabc(env=g, area=area_nodes, nodes_list=nodes_list)

    bn_list.append(best_node)
    bn_metrics.append(best_metric)
    tcd = time.time() - tcs
    print(best_node, best_metric, round(tcd, 2))
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

In [ ]:
# SA
def turn_end_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict, **kwargs):
    print(f'Шаг: {turn} |', f'T={T} |', end='\r')
def best_state_find_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
    print(f'\nШаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |')

mclpsa = BestNodesSA(FirstArrivalUnitState(),
                     metric_demand,
                     end_temperature=0.0001,
                     turns= 1000,
                    #  turn_end_function=turn_end_function,
                     best_state_find_function=best_state_find_function,
                     )


ts = time.time()

best_nodes, best_metric = mclpsa(env=g, area=area_nodes, dynamic_nodes={list(g.nodes())[0]:'best'})
best_node = list(best_nodes.keys())[0]

tf = time.time()
td = round(tf-ts, 2)

print(f'\nВРЕМЯ: {td} сек')


# MCLP

In [ ]:
from genesis.mclp import BestNodesGA, BestNodesKoptG
from genesis.stop_cases import NSameStopCase
from genesis.utils import TimeCheckPoint


main_times_list = {}
main_metrics_list = {}

units_count = 3

for district, scope in scopes.items():
    print(' ')
    print('='*80)
    print(district)
    g = scopes[district][1]
    b = scopes[district][2]
    nodes = ox.graph_to_gdfs(g, edges=False)
    area_nodes = nodes.within(scopes[district][0])
    nodes_list=set(nodes[area_nodes].index)
    start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i}' for i in range(units_count)}
    print('Узлов', sum(area_nodes))
    print('='*80)

    # функция метрики спроса
    metric_demand = Demand(b, buildings_demand_field='kfpo_spros')

    # Списки результатов расчета
    times_list = {}
    metrics_list = {}

    # Расчет SA
    # ============================================================================
    print('\n', '# SA')
    print('-'*80)
    t = TimeCheckPoint()
    t2 = TimeCheckPoint()

    # здесь сам расчет
    def turn_end_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
        print(f'Шаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |', ' '*10, end='\r')
    def best_state_find_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
        print(f'\nШаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |')
    stop_case = NSameStopCase(n=2000)
    mclpsa = BestNodesSA(FirstArrivalUnitState(),
                        metric_demand,
                        end_temperature = 0.0001,
                        turns = 10000,
                        mutation_max_count = 2,
                        turn_end_function=turn_end_function,
                        best_state_find_function = best_state_find_function,
                        stop_case_function = stop_case
                        )
    best_state, best_metric = mclpsa(env=g, area=area_nodes, dynamic_nodes=start_state)

    # завершающая стадия расчета
    time_spent = t()
    print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['SA'] = time_spent
    metrics_list['SA'] = best_metric


    # Расчет GA
    # ============================================================================
    print('\n', '# GA')
    print('-'*80)
    t = TimeCheckPoint()
    # t2 = TimeCheckPoint()

    # здесь сам расчет
    def epoch_end_function(epoch, best_metric, cur_metric, **kwargs):
        print(f'Эпоха: {epoch} |', f'Лучшая метрика: {best_metric} |', f'Текущая метрика: {cur_metric} |', ' '*10, end='\r')
    stop_case = NSameStopCase(n=20)
    mclpga = BestNodesGA(FirstArrivalUnitState(),
                    metric_demand,
                    population_size=50,
                    elite_size=15,
                    epochs=30,
                    mutation_rate=0.5,
                    mutation_max_count=3,
                    epoch_end_function=epoch_end_function,
                    # stop_case_function = stop_case
                    )
    optimal_nodes, best_metric = mclpga(env=g, 
                                  dynamic_nodes=start_state,
                                  area=area_nodes,
                                  )

    # завершающая стадия расчета
    time_spent = t()
    print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GA'] = time_spent
    metrics_list['GA'] = best_metric


    # Расчет GNS
    # ============================================================================
    print('\n', '# GNS')
    print('-'*80)
    t = TimeCheckPoint()
    # t2 = TimeCheckPoint()

    # здесь сам расчет
    def iter_func(i, best_metric, **kwds):
        print(i, ':', best_metric, end='\r')
    bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
                       metric_function=metric_demand,
                       )
    mclpgns = BestNodesKoptG(state_function=FirstArrivalUnitState(),
                       metric_function=metric_demand,
                       best_point_function=bnhc,
                       iterations=5,
                       iter_calc_end_function=iter_func,
                       )
    optimal_nodes, best_metric = mclpgns(env=g,
                        dynamic_nodes=start_state,
                        area=area_nodes,
                        )

    # завершающая стадия расчета
    time_spent = t()
    print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GNS'] = time_spent
    metrics_list['GNS'] = best_metric


    # Расчет GA+GNS
    # ============================================================================
    print('\n', '# GA+GNS')
    print('-'*80)
    t = TimeCheckPoint()
    # t2 = TimeCheckPoint()

    # здесь сам расчет
    def epoch_end_function(epoch, best_metric, cur_metric, **kwargs):
        print(f'Эпоха: {epoch} |', f'Лучшая метрика: {best_metric} |', f'Текущая метрика: {cur_metric} |', ' '*10, end='\r')
    optimal_nodes, best_metric = mclpga(env=g, 
                                  dynamic_nodes=start_state,
                                  area=area_nodes,
                                  )
    optimal_nodes, best_metric = mclpgns(env=g,
                        dynamic_nodes=optimal_nodes,
                        area=area_nodes,
                        )

    # завершающая стадия расчета
    time_spent = t()
    print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GA+GNS'] = time_spent
    metrics_list['GA+GNS'] = best_metric




    # Сохранение итогов в основные словари:
    main_times_list[district+' время'] = times_list
    main_metrics_list[district+' точность'] = metrics_list
    
    # Промежуточное сохранение в DataFrame с объединением main_times_list и main_metrics_list
    # ------------------------------
    tdf = pd.DataFrame({**main_times_list, **main_metrics_list})
    tdf.to_csv('mclp.csv')
    tdf.to_csv(f'mclp {district}.csv')



# LSCP

In [ ]:
from fire_units.metrics import CoverIndexBuilding
from fire_units.mclp import BestNodesGAKopt
from genesis.lscp import LSCPCommon
from genesis.mclp import BestNodesGA, BestNodesKoptG
from genesis.point_selectors import FarNodeSelector, GenesisNodeSelector, RandomNodesSelector
from genesis.stop_cases import MoreEqualStopCase, NSameStopCase
from genesis.utils import TimeCheckPoint


main_times_list = {}
main_metrics_list = {}

units_count = 3

for district, scope in scopes.items():
    print(' ')
    print('='*80)
    print(district)
    g = scopes[district][1]
    b = scopes[district][2]
    nodes = ox.graph_to_gdfs(g, edges=False)
    area_nodes = nodes.within(scopes[district][0])
    nodes_list=set(nodes[area_nodes].index)
    start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i}' for i in range(units_count)}
    print('Узлов', sum(area_nodes))
    print('='*80)

    # функция метрики спроса
    metric_demand = Demand(b, buildings_demand_field='kfpo_spros')
    metric_IPB10 = CoverIndexBuilding(b)

    # Списки результатов расчета
    times_list = {}
    metrics_list = {}


    # # Расчет GNS
    # # ============================================================================
    # print('\n', '# GNS')
    # print('-'*80)
    # t = TimeCheckPoint()
    # # t2 = TimeCheckPoint()

    # здесь сам расчет
    def iter_func(i, best_metric, **kwds):
        print(i, ':', best_metric, end='\r')
    def mclp_end_function(iteration, best_metric, dynamic_nodes, **kwargs):
        print(f'Итерация: {iteration} |', f'Лучшая метрика: {best_metric} |', f'Подразделений: {len(dynamic_nodes)}')

    # bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
    #                 #    metric_function=metric_demand,
    #                    metric_function=metric_IPB10,                       
    #                    )
    # mclpgns = BestNodesKoptG(state_function=FirstArrivalUnitState(),
    #                 #    metric_function=metric_demand,
    #                 #    metric_function=CoverIndex(5),
    #                    metric_function=metric_IPB10, 
    #                    best_point_function=bnhc,
    #                    iterations=5,
    #                    iter_calc_end_function=iter_func,
    #                    )

    # # Случай останоски
    # stop_case = MoreEqualStopCase(95)

    # # Селектор следующей точки
    # # point_selector = RandomNodesSelector()
    # # point_selector = GenesisNodeSelector(state_function = FirstArrivalUnitState(),
    # #                                     metric_function = metric_demand)
    # point_selector = FarNodeSelector(FirstArrivalUnitState())
    
    # # Сборка алгоритма
    # genesis = LSCPCommon(mclp_function = mclpgns,
    #                  point_selector = point_selector,
    #                 stop_case_function = stop_case,
    #                 start_names_index = 5,
    #                 names_pattern='ПСЧ {}',
    #                 after_mclp_function=mclp_end_function,
    #                 )
    # # Собственно применения построенного алгоритма
    # best_dynamic_nodes, best_metric = genesis(env = g,
    #                             dynamic_nodes = start_state,
    #                             area=area_nodes,
    #                             # static_nodes = existed_units,
    #                             )

    # # завершающая стадия расчета
    # time_spent = t()
    # print('\n')
    # print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    # print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # # Сохраняем результат вычисления
    # times_list['GNS'] = time_spent
    # metrics_list['GNS'] = best_metric


    # Расчет GA+GNS
    # ============================================================================
    print('\n', '# GNS')
    print('-'*80)
    t = TimeCheckPoint()
    # t2 = TimeCheckPoint()

    # здесь сам расчет
    # # Метрика: ИПЗ-10
    # IPB10 = CoverIndexBuilding(b)

    # Функция оптимизации k узлов методом hillclimber
    bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
                        metric_function = metric_IPB10,
                        )

    kopt = BestNodesKoptG(state_function=FirstArrivalUnitState(),
                        metric_function = metric_IPB10,
                        best_point_function=bnhc,
                        iterations=3,
                        iter_calc_end_function=iter_func,
                        )


    mclp = BestNodesGAKopt(state_function = FirstArrivalUnitState(),
                        # metric_function = ArrivalTime(),            # Оптимизируем среднее время прибытия в любой узел
                        metric_function = metric_IPB10,                        # Оптимизируем ИПЗ-10
                        kopt_function = kopt,
                        population_size=50,
                        epochs = 20,
                        elite_size=15,
                        epoch_end_function = epoch_end_function,
                        mutation_max_count = 2,
                        stop_case_function = NSameStopCase(15)      # Условие остановки не менее N одинаковых метрик
                        )

    # Критерий остановки
    stop_case = MoreEqualStopCase(98)

    # Селектор следующей точки
    point_selector = RandomNodesSelector()

    # Сборка алгоритма
    genesis = timing(LSCPCommon(mclp_function = mclp,
                        point_selector = point_selector,
                        stop_case_function = stop_case,
                        start_names_index = 12,
                        names_pattern='псч {}',
                        after_mclp_function = mclp_end_function,
                        ))

    # Расчет только динамические узлы
    best_dynamic_nodes, best_metric = genesis(env = g,
                                dynamic_nodes = start_state,
                                # dynamic_nodes = best_dynamic_nodes,     # Продолжение расчета с предыдущего шага                           
                                area = area_nodes,
                                )

    # завершающая стадия расчета
    time_spent = t()
    print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}', 'Подразделений:', len(best_dynamic_nodes))
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GNS'] = time_spent
    metrics_list['GNS'] = best_metric



    # Сохранение итогов в основные словари:
    main_times_list[district+' время'] = times_list
    main_metrics_list[district+' точность'] = metrics_list
    
    # Промежуточное сохранение в DataFrame с объединением main_times_list и main_metrics_list
    # ------------------------------
    tdf = pd.DataFrame({**main_times_list, **main_metrics_list})
    tdf.to_csv('lscp.csv')
    tdf.to_csv(f'lscp {district}.csv')

# Черновики

In [ ]:
# Печать карты
def plot_scope_result(g:nx.MultiDiGraph,
                      district:gpd.GeoDataFrame,
                      point:list,
                      buildings:pd.DataFrame,
                      ax=None):
    nodes = ox.graph_to_gdfs(g, edges=False)
    if ax is None:
        fig, ax = ox.plot_graph(g, edge_linewidth=0.2, node_size=0, show=False, bgcolor='white')
    district.boundary.plot(ax=ax, color='r')
    buildings.plot(ax=ax)
    nodes.loc[point].plot(ax=ax, markersize=25, color='red')

    return 1, 2

d = timing(plot_scope_result(g, districts.loc[[district]], [best_node], b))
d

In [ ]:
state, _ = FirstArrivalUnitState()(g, points={best_node:'best'})
area = area_nodes
if isinstance(state, pd.Series):
    state_c = state.loc[area]
else:
    state_c = [x for x in state if x in area]

# Расчет метрики по спросу
merged_df = pd.merge(b,
                    state_c,
                    left_on='node',
                    right_index=True)

merged_df['demand_val'] = merged_df['node'] / merged_df['times']

In [ ]:
def plot_scope_result(g:nx.MultiDiGraph,
                      district:gpd.GeoDataFrame,
                      point:list,
                      buildings:pd.DataFrame,
                      ax=None):
    nodes = ox.graph_to_gdfs(g, edges=False)
    if ax is None:
        fig, ax = ox.plot_graph(g, edge_linewidth=0.2, node_size=0, show=False, bgcolor='white')
    district.boundary.plot(ax=ax, color='r')
    buildings.plot(ax=ax, column='times', cmap=CMAP_GrGldRd, legend=True)
    nodes.loc[point].plot(ax=ax, markersize=25, color='red')

plot_scope_result(g, districts.loc[[district]], [best_node], merged_df)

#### Тест HC с исследованием областей

In [ ]:
bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)

bn_list = []
bn_metrics = []
i=1

size=200
split_list = [list(nodes_list)[i:i+size] for i in range(0, len(nodes_list), size)]

ts = time.time()
while time.time() - ts < etalon_time:
    tcs = time.time()
    best_cur_node = None
    best_cur_metric = 0
    j=1
    for nodes_part in split_list:
        start_node = random.choice(nodes_part)
        best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
        if best_metric > best_cur_metric:
            best_cur_node = best_node
            best_cur_metric = best_metric
        print(f'{i}:', j, 'из', len(split_list), best_cur_metric, end='\r')
        j+=1

    bn_list.append(best_cur_node)
    bn_metrics.append(best_cur_metric)
    tcd = time.time() - tcs
    print('\n', i, best_cur_node, best_cur_metric, round(tcd, 2), ' '*10) #, end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

In [ ]:
random.choices(list(nodes_list), k=5)

In [ ]:
# from genesis.best_points import NodeMetric
# from genesis.core import StateBase


# class BestNodeBee(BestNodeHillClimbing):
#     '''
#         Поиск лучших узлов графа с использованием алгоритма пчелиной колонии
#         (Artificial Bee Colony Optimization, ABC).
#         Могут быть возвращены только узлы из которых можно попасть в большую часть 
#         других узлов графа.
#         Если таковых узлов нет, возвращается ошибка некорректности графа.
#         (граф должен быть проверен на корректность прежде чем будет передан функции)

#         Узлы для которых метрика не может быть вычислена (например слабо связанные
#         с основным графом) не учитываются.

#         ## Область применения
#         Определение размещения одного узла с наилучшими показателями.
#         Дает достаточно точное решение. Хорошо подходит для больших графов.
#     '''
#     def __init__(self,
#                  state_function: StateBase,
#                  metric_function: MetricBase,
#                  appr_val: float = 0.95,
#                  all_neighbors: bool=True,
#                  scouts_count: int = 100,
#                  best_scouts_count: int = 5,
#                 #  after_local_jump_function: callable = None,
#                  **kwargs) -> None:
#         '''
#         ## Аргументы
#         `state_function`: StateBase
#             функция расчета состояния окружения
#         `metric_function`: MetricBase
#             Функция расчета метрики
#         `appr_val`: = 0.95
#             Доля узлов графа, покрытие которой считается приемлемой для принятия расчетной метрики. 
#             Если при расчете метрик, из стартового узла (узлов) достижимо меньшее количество узлов,
#             то такой узел не рассматривается.
#         `all_neighbors`: bool=True
#             Если True рассматриваются все узлы смежные с рассчитываемым узлом.
#             Если False - только исходящие.
#         `scouts_count`: int=100
#             Количество пчел-разведчиков.
#         `best_scouts_count`: int=5
#             Количество лучших пчел-разведчиков.
#             Для них производится дальнейшая оптимизация.
        
#         '''
#         self.node_metric_func = NodeMetric(state_function,
#                                            metric_function,
#                                            appr_val,
#                                            err_val=None,
#                                            **kwargs)
#         self.scouts_count = scouts_count
#         self.best_scouts_count = best_scouts_count
#         super().__init__(state_function,
#                          metric_function,
#                          appr_val=appr_val,
#                          all_neighbors=all_neighbors,
#                          **kwargs)


#     def __call__(self,
#                  env:nx.MultiDiGraph,
#                  area=None,
#                  nodes_list:set=None,
#                  **kwargs) -> tuple[int | None, float | None]:
#         '''
#         Реализация: `scouts_count` пчел-разведчиков случайным образом проверяют узлы
        
#         при помощи BestNodesHillClimbing ищется лучшая точка, 
#         после чего обезьяна прыгает по ближайшим вершинам и вновь использует BestNodesHillClimbing.

#         Так, пока не будет найден глобальный оптимум или не будет достигнуто количество итераций.

#         ## Аргументы
#         `env`:nx.MultiDiGraph
#             Граф улично-дорожной сети
#         `area`: pd.Series = None
#             Маска узлов графа. Значениями True отмечены узлы графа - цели расчета леса Вороного.
#             Если не указана, расчет производится для всех узлов графа.
#         `nodes_list`:set=None
#             Множество узлов графа которые будут рассмотрены в качестве кандидатов.
#             Если не указан, то будут рассмотрены все узлы графа.

#         ## Возвращает
#         `best_node`: int, `best_metric`: float
#             Лучший узел, лучшая метрика
#         '''

#         if not isinstance(env, nx.MultiDiGraph):
#             raise TypeError("Тип переменной `env` должен быть MultiDiGraph!")
#         if not area is None and not isinstance(area, pd.Series):
#             raise TypeError(f'Аргумент `area` должен иметь тип `pd.Series`! Имеет {type(area)}')

#         # Если списка узлов изначально не передано, рассматриваются все узлы графа
#         if nodes_list is None:
#             nodes_list = env.nodes()

#         # 1. Разведка местности
#         t_list = []
#         t_metric = []
#         for start_node in random.choices(list(nodes_list), k=self.scouts_count):
#             start_metric = self.node_metric_func(env=env, area=area, node=start_node)
#             t_list.append(start_node)
#             t_metric.append(start_metric)
        
#         # 2. Выбираем результаты лучших пчел-разведчиков
#         tdf = pd.DataFrame({'node':t_list, 'metric':t_metric})
#         if self.metric_function.compare(1,2)==2:
#             tdf = tdf.sort_values('metric', ascending=False)
#         else:
#             tdf = tdf.sort_values('metric', ascending=True)
#         # tdf.sort_values('metric', ascending=self.metric_function.compare(1,2)==2)
#         # print(tdf['metric'][:10])

#         # 3. Для каждого из лучших результатов пытаемся найти еще лучшие значения метрики в окрестности
#         best_node = None
#         best_metric  = None
#         for node in tdf['node'][:self.best_scouts_count]:
#             cur_node, cur_metric = super().__call__(
#                     env=env,
#                     start_node=node,
#                     area=area,
#                     **kwargs,
#                     )

#             if best_node is None:
#                 best_node, best_metric = cur_node, cur_metric
#             else:
#                 if best_node != cur_node and \
#                         self.metric_function.compare(best_metric, cur_metric) == cur_metric:
#                     best_node, best_metric = cur_node, cur_metric

#         return best_node, best_metric

In [ ]:
from genesis.best_points import BestNodeBee

bnabc = BestNodeBee(FirstArrivalUnitState(),
                    metric_demand,
                    # scouts_count=150,
                    # best_scouts_count=3,
                    )

ts = time.time()
bn_list = []
bn_metrics = []
i=1
while time.time() - ts < etalon_time:
# while i < 5:
    print(i, end=':')
    tcs = time.time()

    best_node, best_metric = bnabc(env=g, area=area_nodes, nodes_list=nodes_list)

    bn_list.append(best_node)
    bn_metrics.append(best_metric)
    tcd = time.time() - tcs
    print(best_node, best_metric, round(tcd, 2))
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

In [ ]:
from genesis.best_points import NodeMetric


bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)
node_metric = NodeMetric(FirstArrivalUnitState(), metric_demand)


bn_list = []
bn_metrics = []
i=1

k = 100
k_best = 5

ts = time.time()
while time.time() - ts < etalon_time:
# while i < 10:
    tcs = time.time()
    best_cur_node = None
    best_cur_metric = 0

    t_list = []
    t_metric = []
    j=1
    for start_node in random.choices(list(nodes_list), k=k):
        start_metric = node_metric(env=g, area=area_nodes, node=start_node)
        if start_metric > best_cur_metric:
            best_cur_node = start_node
            best_cur_metric = start_metric
        t_list.append(start_node)
        t_metric.append(start_metric)

        print(f'{i}:', j, 'из', k, best_cur_metric, start_metric,  end='\r')
        j+=1

    tdf = pd.DataFrame({'node':t_list, 'metric':t_metric}).sort_values('metric', ascending=False)
    
    best_cur_node = None
    best_cur_metric = 0
    j=1
    for node in tdf['node'][:k_best]:
        best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=node)
        if best_metric > best_cur_metric:
            best_cur_node = best_node
            best_cur_metric = best_metric
        print(f'{i}:', j, 'из', k_best, best_cur_metric, end='\r')
        j+=1

    bn_list.append(best_cur_node)
    bn_metrics.append(best_cur_metric)
    tcd = time.time() - tcs
    print('\n', i, best_cur_node, best_cur_metric, round(tcd, 2), ' '*10) #, end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

    ВРЕМЯ: 347.34 сек
    КОРРЕКТНОСТЬ: 17 из 20
    ВРЕМЯ ОДНОГО РАСЧЕТА: 17.366999999999997 сек

In [ ]:
size=300
split_list = [list(nodes_list)[i:i+size] for i in range(0, len(nodes_list), size)]
for sl in split_list:
    print(len(sl))